In [ ]:
pip install datasets


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 491.5/491.5 kB 24.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.3/116.3 kB 12.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 193.6/193.6 kB 16.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.5/143.5 kB 13.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 194.8/194.8 kB 16.6 MB/s eta 0:00:00
  Attempting uninstall: fsspec
    Found existing installation: fsspec 2025.3.2
    Uninstalling fsspec-2025.3.2:
      Successfully uninstalled fsspec-2025.3.2
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
torch 2.6.0+cu124 requires nvidia-cublas-cu12==12.4.5.8; platform_system == "Linux" and platform_machine == "x86_64", but you have nvidia-cublas-cu12 12.5.3.2 which is incompatible.
torch 2.6.0+cu124 requires nvidia-cuda-cupti-cu12==12.4.127; platform_system 

In [ ]:
from datasets import load_dataset
import pandas as pd
import random
from transformers import MarianMTModel, MarianTokenizer
import torch
from tqdm import tqdm

# Загружаем датасет
ds = load_dataset("LuangMV97/Empathetic_counseling_Dataset", split="train")
samples = ds.shuffle(seed=42).select(range(2000))

# Загружаем модель перевода
model_name = "Helsinki-NLP/opus-mt-en-ru"
tokenizer = MarianTokenizer.from_pretrained(model_name)
model = MarianMTModel.from_pretrained(model_name)
# Explicitly move the model to the CUDA device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)
model.eval()


def translate_batch(texts):
    batch = tokenizer(texts, return_tensors="pt", padding=True, truncation=True)
    # Move all tensors in the batch to the specified device
    batch = {k: v.to(device) for k, v in batch.items()}
    with torch.no_grad():
        outputs = model.generate(**batch)
    return [tokenizer.decode(t, skip_special_tokens=True) for t in outputs]


/usr/local/lib/python3.11/dist-packages/transformers/models/marian/tokenization_marian.py:175: UserWarning: Recommended: pip install sacremoses.
  warnings.warn("Recommended: pip install sacremoses.")


In [ ]:
# Переводим пары input → label
translated_data = []
batch_size = 16
for i in tqdm(range(0, len(samples), batch_size)):
    batch = samples.select(range(i, min(i + batch_size, len(samples))))
    prompts = batch["input"]
    responses = batch["label"]
    ru_prompts = translate_batch(prompts)
    ru_responses = translate_batch(responses)
    for p, r in zip(ru_prompts, ru_responses):
        translated_data.append({"prompt": p, "response": r})

# Сохраняем
df = pd.DataFrame(translated_data)
df.to_json("empathy_ru_pairs.jsonl", orient="records", lines=True, force_ascii=False)

print(" Всё готово! Переведённые пары сохранены.")

In [ ]:
import pandas as pd

# Загружаем переведённые пары
df = pd.read_json("empathy_ru_pairs.jsonl", lines=True)

# Удаляем пустые строки и одинаковые prompt-response
df = df.dropna(subset=["prompt", "response"])
df = df[df["prompt"].str.strip() != ""]
df = df[df["response"].str.strip() != ""]
df = df[df["prompt"] != df["response"]]

# Удаляем слишком короткие (менее 3 слов)
df = df[df["prompt"].str.split().str.len() >= 3]
df = df[df["response"].str.split().str.len() >= 3]

# (по желанию) можно ограничить длину в символах
df = df[df["prompt"].str.len() <= 250]
df = df[df["response"].str.len() <= 250]

# Сохраняем очищенный датасет
df.to_json("empathy_ru_pairs_cleaned.jsonl", orient="records", lines=True, force_ascii=False)

print(f" Готово! Итоговая выборка: {len(df)} строк. Сохранили в empathy_ru_pairs_cleaned.jsonl")
